In [1]:
import sys
sys.path.append("/Users/avelezxerenity/Documents/GitHub/pysdk")
import os
from inflation_query.Inflation_query import implied_inflation_calc
from src.xerenity.xty import Xerenity
from utilities.date_functions import add_months,ql_to_datetime
from swap_functions.ibr_quantlib_details import ibr_quantlib_det,ibr_overnight_index,ibr_swap_cupon_helper,depo_helpers_ibr
import pandas as pd
import QuantLib as ql
import plotly.graph_objects as go
from datetime import datetime,date
xty = Xerenity(
    username=os.getenv('XTY_USER'),
    password=os.getenv('XTY_PWD'),
)


def nom_to_effective(nominal_rate,compounding_frequency):
    return (1 + nominal_rate / compounding_frequency) ** compounding_frequency - 1
#calc_date = ql.Date(2, 11, 2023)
#ql.Settings.instance().evaluationDate = calc_date

2023-11-07 20:46:08,986:INFO - HTTP Request: POST https://tvpehjbqxpiswkqszwwv.supabase.co/auth/v1/token?grant_type=password "HTTP/1.1 200 OK"


<QuantLib.QuantLib.SwapRateHelper; proxy of <Swig Object of type 'ext::shared_ptr< SwapRateHelper > *' at 0x13f0854a0> >


2023-11-07 20:46:09,702:INFO - HTTP Request: POST https://tvpehjbqxpiswkqszwwv.supabase.co/auth/v1/token?grant_type=password "HTTP/1.1 200 OK"


In [2]:
ibr_on=xty.BanRep().get_econ_data_last(id_serie=19).data[0]['valor']/100


today=date.today()
init_date=datetime(2023, 11, 7).date()


final_date=init_date
ibr_data=pd.DataFrame(xty.get_date_range(table_name='ibr_swaps_cluster',date_column_name='execution_timestamp').data)
ibr_data['execution_timestamp'] = pd.to_datetime(ibr_data['execution_timestamp']).dt.date
mask = (ibr_data['execution_timestamp'] >= init_date) & (ibr_data['execution_timestamp'] <= final_date) & (ibr_data['action_type'] == 'NEWT')
ibr_data = ibr_data[mask] 
ibr_data= ibr_data[abs(ibr_data['days_diff_trade_effe']) < 7]
ibr_cluster_mean = pd.DataFrame(ibr_data.groupby('month_diff_effective_expiration')['rate'].mean())
ibr_cluster_mean


OIS_helpers = []
for row in ibr_cluster_mean.iterrows():

    if row[0]<=18:
        OIS_helpers.append(depo_helpers_ibr(row[1][0],row[0],ql.Months))

    else:
        OIS_helpers.append( ibr_swap_cupon_helper(row[1][0],int(row[0]/12),ql.Years))
        print(row[1][0])
        print(int(row[0]/12))
OIS_helpers.append(depo_helpers_ibr(ibr_on,8,ql.Days))       



settlement_day=ql.Date.todaysDate()+ql.Period(5,ql.Days)
print(settlement_day)
ql.Settings.instance().evaluationDate = settlement_day
#curve = ql.PiecewiseLinearZero(0, ibr_quantlib_det['calendar'], OIS_helpers, ql.Actual360())
curve =ql.PiecewiseSplineCubicDiscount(0, ibr_quantlib_det['calendar'], OIS_helpers, ql.Actual360())
#curve_ff = ql.FlatForward(0, ibr_quantlib_det['calendar'], OIS_helpers, ql.Actual360(), ql.Linear())
#curve = ql.Log(ql.NullCalendar(), OIS_helpers, ql.Actual360())




start_date = ql.Date(15, 12, 2023)


# Define the forward rate tenor
forward_tenor_1m = ql.Period(1, ql.Months)
forward_tenor_1y = ql.Period(1, ql.Years)

# Initialize lists to store results
dates = []
forward_rates = []
# Loop through 1-year steps up to 10 years (120 months)
for i in range(1, 119):
    try:
        # Calculate the forward rate for the current step
        first_date = start_date + ql.Period(i, ql.Months)
        end_date = first_date + ql.Period(3, ql.Months)
        forward_rate = curve.forwardRate(first_date, end_date, ql.Actual360(), ql.Compounded).rate()
        dates.append(first_date)
        forward_rates.append(forward_rate)
        # Print the result
        #print(f"1-month {i/12} year forward rate: {forward_rate:.4%}")
    except:
        
        print('tenor to calculate not un curve helper')
df = pd.DataFrame(list(zip(dates, forward_rates)), columns=['Maturity Date', 'rate'])
df['Maturity Date'] = df['Maturity Date'].apply(ql_to_datetime)
df.set_index('Maturity Date', inplace=True)


ibr_on=pd.DataFrame(xty.BanRep().get_econ_data_last_n(id_serie=15,n=15*365).data).drop('id_serie',axis=1)
ibr_on['fecha'] = pd.to_datetime(ibr_on['fecha'])
ibr_on.set_index('fecha', inplace=True)


2023-11-07 20:46:10,152:INFO - HTTP Request: GET https://tvpehjbqxpiswkqszwwv.supabase.co/rest/v1/banrep_serie_value?select=%2A&id_serie=eq.19&order=fecha.desc&limit=1 "HTTP/1.1 200 OK"
2023-11-07 20:46:10,405:INFO - HTTP Request: GET https://tvpehjbqxpiswkqszwwv.supabase.co/rest/v1/ibr_swaps_cluster?select=%2A "HTTP/1.1 200 OK"
/Users/avelezxerenity/Documents/GitHub/pysdk/src/xerenity/xty.py:133: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[column] = pd.to_datetime(df[column])
/Users/avelezxerenity/Documents/GitHub/pysdk/src/xerenity/xty.py:133: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[column] = pd.to_datetime(df[column])
2023-11-07 20:46:10,683:INFO - HTTP Request: GET https://tvpehjbqxpiswkqszwwv

0.09462000000000001
2
0.0877
3
0.0843
4
0.084086125
5
0.083875
10
November 12th, 2023
tenor to calculate not un curve helper
tenor to calculate not un curve helper


2023-11-07 20:46:11,105:INFO - HTTP Request: GET https://tvpehjbqxpiswkqszwwv.supabase.co/rest/v1/banrep_serie_value?select=%2A&id_serie=eq.15&order=fecha.desc&limit=5475 "HTTP/1.1 200 OK"


In [3]:
df

,rate
Maturity Date,
2024-01-15,0.125318
2024-02-15,0.122502
2024-03-15,0.119115
2024-04-15,0.115229
2024-05-15,0.111079
...,...
2033-04-15,0.090618
2033-05-15,0.091205
2033-06-15,0.091833


In [4]:
cv_tasa=pd.read_csv('historic_vis.csv')
cv_tasa.set_index('fecha', inplace=True)

# Assuming df_tpm and df_cv_tasa are your DataFrames
ibr_on.index = pd.to_datetime(ibr_on.index)
cv_tasa.index = pd.to_datetime(cv_tasa.index)
# Resample TPM to daily and forward fill to fill missing values
df_tpm_resampled = ibr_on.resample('D').ffill()

# Merge CV_TASA with TPM using merge_asof
result_df = pd.merge_asof(cv_tasa, df_tpm_resampled, left_index=True, right_index=True, direction='nearest', suffixes=('tasa','valor'))

FileNotFoundError: [Errno 2] No such file or directory: 'historic_vis.csv'

In [ ]:
result_df

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Drop rows with NaN values
result_df = result_df.dropna(subset=['tasa','valor'])

# Perform linear regression with numpy
X = result_df['valor'].values
y = result_df['tasa'].values

# Add a column of ones for the constant term
X = np.column_stack((np.ones_like(X), X))

# Calculate the coefficients
beta = np.linalg.lstsq(X, y, rcond=None)[0]

# Plot the data and regression line
plt.scatter(result_df['valor'], result_df['tasa'], label='Data')
plt.plot(result_df['valor'], beta[0] + beta[1] * result_df['valor'], color='red', label='Linear Regression')

# Add labels and move legend to the lower right corner
plt.xlabel('IBR-on')
plt.ylabel('Credito vivienda')
plt.legend(loc='lower right')

# Display regression results in a box
plt.text(0.05, 0.9, f'Intercept: {beta[0]:.4f}\nSlope: {beta[1]:.4f}', transform=plt.gca().transAxes, bbox=dict(facecolor='white', alpha=0.8))

# Show the plot
plt.show()

def predict_valor_CV_TASA(ibr_on,slope,intercept):
    # Coefficients from the linear regression
    # intercept = 0.1234
    # slope = 0.5678

    # Use the formula to predict valor_CV_TASA
    predicted_valor_CV_TASA = intercept + slope * ibr_on

    return predicted_valor_CV_TASA





In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=ibr_on['valor'].index, 
                         y=ibr_on['valor'],
                         mode='lines',
                         name='IBR-on historico e.a'))


fig.add_trace(go.Scatter(x=df['rate'].index, 
                        y=nom_to_effective(df['rate'],365)*100,
                        mode='lines',
                        name='IBR-on implicito e.a'))

fig.add_trace(go.Scatter(x=cv_tasa.index, 
                        y=cv_tasa['tasa'],
                        mode='lines',
                        name='Serie de creditos de vivienda VIS -Banco de la republica '))
# Add traces for dates after today
fig.add_trace(go.Scatter(x=df.index, 
                        y=predict_valor_CV_TASA(nom_to_effective(df['rate'],365)*100,beta[1],beta[0]),
                        mode='lines',
                        name='Creditos de vivienda implicito e.a'))


# Update layout
fig.update_layout(
    title='Tasas IBR overnight y Creditos de vivienda VIS ',
    xaxis=dict(title='Fechas'),
    yaxis=dict(title='Tasa e.a'),
)
# Show the plot
#fig.show()
fig.write_html('Ibr-on VIS.html')

In [ ]:
#ibr_on.to_csv('ibr_ea_historica.csv', index=True)
#df.to_csv('ibr_ea_implicita.csv', index=True)
cv_tasa.to_csv('credito_vivienda_vis_ea_historica.csv', index=True)
predict_valor_CV_TASA(nom_to_effective(df['rate'],365)*100,beta[1],beta[0]).to_csv('credito_vivienda_vis_ea_implicita.csv', index=True)

In [ ]:

# Assuming cv_tasa and df_tpm_resampled are your DataFrames
cv_tasa.index = pd.to_datetime(cv_tasa.index)

# Merge CV_TASA with TPM using merge_asof
result_df = pd.merge_asof(cv_tasa, df_tpm_resampled, left_index=True, right_index=True, direction='nearest', suffixes=('_CV_TASA', '_TPM'))

# Drop rows with NaN values
result_df = result_df.dropna(subset=['valor_TPM', 'valor_CV_TASA'])

# Perform linear regression with numpy
X = result_df['valor_TPM'].values
y = result_df['valor_CV_TASA'].values

# Add a column of ones for the constant term
X = np.column_stack((np.ones_like(X), X))

# Calculate the coefficients
beta = np.linalg.lstsq(X, y, rcond=None)[0]

# Plot the data and regression line
plt.scatter(result_df['valor_TPM'], result_df['valor_CV_TASA'], label='Data')
plt.plot(result_df['valor_TPM'], beta[0] + beta[1] * result_df['valor_TPM'], color='red', label='Linear Regression')

# Add labels and move legend to the lower right corner
plt.xlabel('IBR overnigth')
plt.ylabel('Creditos de vivienda')
plt.legend(loc='lower right')

# Display regression results in a box
plt.text(0.05, 0.9, f'Intercept: {beta[0]:.4f}\nSlope: {beta[1]:.4f}', transform=plt.gca().transAxes, bbox=dict(facecolor='white', alpha=0.8))

# Show the plot
plt.show()






